# Turnover Prediction — Alternatif Metode (v2)

Notebook ini melatih **Employee Turnover Prediction** dengan pendekatan yang **berbeda dari v1**.

| | v1 (`train_model.ipynb`) | v2 (notebook ini) |
|---|---|---|
| Algoritma | Logistic Regression (linear, 1 model) | 7 kandidat: linear, tree ensemble, boosting, SVM |
| Boosting | — | XGBoost, LightGBM, CatBoost, HistGradientBoosting |
| Ensemble | — | Soft Voting (gabungan model terbaik) |
| Kalibrasi | bawaan (sudah terkalibrasi) | dicek Brier + kurva kalibrasi per model |
| Interpretasi | permutation importance + PDP | permutation importance + **SHAP** |
| Output | `models/v1/` | `models/v2/` (metadata tetap kompatibel API) |

## Alur kerja

| # | Tahap | Yang dikerjakan |
|---|-------|-----------------|
| 1 | Load & audit | data sama dengan v1 agar perbandingan adil |
| 2 | Feature engineering | **fungsi `buat_fitur()` identik** dengan v1 & `hr_intelligence_api` |
| 3 | Split | stratified hold-out 80/20, `random_state` sama dengan v1 |
| 4 | Perbandingan CV | 7 model dibandingkan dengan `RepeatedStratifiedKFold` |
| 5 | Tuning | `RandomizedSearchCV` untuk model teratas |
| 6 | Ensemble | soft-voting dari model terbaik |
| 7 | Pemilihan juara | berdasarkan ROC-AUC CV (bukan test) |
| 8 | Threshold | dipilih dari prediksi out-of-fold data train |
| 9 | Evaluasi final | ROC-AUC, PR-AUC, Brier, kalibrasi, confusion matrix, 95% CI |
| 10 | Interpretasi | permutation importance + SHAP |
| 11 | Simpan | `models/v2/{turnover_model.joblib, model_metadata.json}` |

## Prinsip yang dijaga

- **Adil:** split, fitur, dan definisi metrik sama persis dengan v1 — perbedaan hasil murni dari algoritma.
- **Tanpa leakage:** seluruh preprocessing berada di dalam `Pipeline` dan hanya dilatih di data train.
- **Test disentuh sekali** untuk evaluasi akhir; pemilihan model & threshold memakai CV/OOF.
- **Kontrak API tetap terjaga:** nama fitur, urutan kolom, dan `tanggal_acuan` tidak berubah.
- **Reproducible:** seed tetap, versi library & hyperparameter dicatat di metadata.

> **Library:** `pip install pandas numpy scikit-learn scipy joblib matplotlib seaborn xgboost lightgbm catboost shap`

## 0. Import & Konfigurasi

In [1]:
import json
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from scipy.stats import loguniform, randint
from sklearn.base import clone
from sklearn.calibration import calibration_curve
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
    StackingClassifier,
    VotingClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings("ignore")

SEED = 42
np.random.seed(SEED)

# --- Lokasi data (dicari relatif terhadap cwd) ---
CANDIDATE_PATHS = [
    "hr_turnover.csv",
    "../data/hr_turnover.csv",
    "data/hr_turnover.csv",
]
DATA_PATH = next((Path(p) for p in CANDIDATE_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "hr_turnover.csv tidak ditemukan. Jalankan notebook dari folder "
        "notebooks/ atau root repo, atau sesuaikan CANDIDATE_PATHS."
    )

# --- Root repo & folder artifact v2 ---
CWD = Path.cwd()
PROJECT_ROOT = CWD if (CWD / "data").exists() else CWD.parent
ARTIFACT_DIR = PROJECT_ROOT / "models" / "v2"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_VERSION = "v2"

# --- Kontrak fitur (SAMA dengan v1 & hr_intelligence_api) ---
TARGET = "target_turnover"
ID_COL = "employee_id"
DATE_COLS = ["tanggal_masuk", "tanggal_promosi"]
REF_DATE = pd.Timestamp("2025-01-01")

TEST_SIZE = 0.20
REFIT_ON_FULL_DATA = True
TUNE_TOP_N = 2  # jumlah model teratas yang di-tuning

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 170)
sns.set_theme(style="whitegrid", context="notebook")

print("Data        :", DATA_PATH)
print("Artifact dir:", ARTIFACT_DIR)
print("pandas      :", pd.__version__)
print("sklearn     :", sklearn.__version__)

Data        : data/hr_turnover.csv
Artifact dir: /workspace/project/hr-intelligence-system/models/v2
pandas      : 3.0.6
sklearn     : 1.9.1


## 1. Load & Audit Data

Audit ini sama dengan v1 supaya perbandingan metode benar-benar apple-to-apple.

In [2]:
df_raw = pd.read_csv(DATA_PATH, parse_dates=DATE_COLS)

audit = pd.Series({
    "jumlah baris": len(df_raw),
    "baris duplikat": int(df_raw.duplicated().sum()),
    "employee_id unik": bool(df_raw[ID_COL].is_unique),
    "missing tanggal_promosi": int(df_raw["tanggal_promosi"].isna().sum()),
    "rasio turnover": round(float(df_raw[TARGET].mean()), 4),
})
print(audit.to_string())
df_raw.head()

jumlah baris                1000
baris duplikat                 0
employee_id unik            True
missing tanggal_promosi      582
rasio turnover             0.356


,employee_id,usia,masa_kerja_bulan,jam_lembur_bulanan,gaji_bulanan,tanggal_masuk,tanggal_promosi,departemen,status_kerja,pendidikan,target_turnover
0,EMP0001,38,172,2,14514816,2010-08-12,2015-07-28,Marketing,Tetap,S1,0
1,EMP0002,30,39,0,5985038,2021-09-05,NaT,Sales,Tetap,SMA,0
2,EMP0003,32,99,9,5650960,2016-09-10,NaT,Legal,Freelance,SMA,0
3,EMP0004,32,20,0,15660028,2023-04-12,2024-10-01,HR,Tetap,S2,0
4,EMP0005,34,132,16,7245406,2013-12-06,2015-07-11,Finance,Tetap,SMA,0


## 2. Feature Engineering & Split

`buat_fitur()` di bawah **identik** dengan v1 dan dengan `hr_intelligence_api/app/features/features.py`.
Ini penting: model v2 harus bisa dilayani API tanpa mengubah kontrak fitur.

Split memakai `test_size=0.20`, `stratify=y`, dan `random_state=42` — sama dengan v1,
sehingga perbedaan performa murni berasal dari algoritma.

In [3]:
NUM_COLS = [
    "usia", "masa_kerja_bulan", "jam_lembur_bulanan", "log_gaji",
    "usia_saat_masuk", "ada_promosi", "bulan_sejak_promosi", "bulan_hingga_promosi",
]
CAT_COLS = ["departemen", "status_kerja", "pendidikan"]
FEATURES = NUM_COLS + CAT_COLS


def buat_fitur(d: pd.DataFrame, ref_date: pd.Timestamp = REF_DATE) -> pd.DataFrame:
    """Ubah data mentah (kolom seperti CSV) menjadi fitur model."""
    d = d.copy()
    masuk = pd.to_datetime(d["tanggal_masuk"])
    promosi = pd.to_datetime(d["tanggal_promosi"])

    d["ada_promosi"] = promosi.notna().astype(float)
    d["bulan_sejak_promosi"] = (ref_date - promosi).dt.days / 30.44
    d["bulan_hingga_promosi"] = (promosi - masuk).dt.days / 30.44
    d["usia_saat_masuk"] = d["usia"] - d["masa_kerja_bulan"] / 12
    d["log_gaji"] = np.log1p(d["gaji_bulanan"])

    fitur = d[FEATURES].copy()
    fitur[NUM_COLS] = fitur[NUM_COLS].astype(float)
    return fitur


def make_preprocessor() -> ColumnTransformer:
    numeric = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical = OneHotEncoder(handle_unknown="ignore")
    return ColumnTransformer([
        ("num", numeric, NUM_COLS),
        ("cat", categorical, CAT_COLS),
    ])


X = buat_fitur(df_raw)
y = df_raw[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=SEED
)

print("Bentuk X      :", X.shape, "| jumlah fitur:", len(FEATURES))
print("Train / Test  :", X_train.shape[0], "/", X_test.shape[0])
print("Rasio turnover: train", round(y_train.mean(), 3), "| test", round(y_test.mean(), 3))

Bentuk X      : (1000, 11) | jumlah fitur: 11
Train / Test  : 800 / 200
Rasio turnover: train 0.356 | test 0.355


## 3. Kandidat Model (Metode Selain Logistic Regression)

Semua kandidat memakai **preprocessor yang sama**. Tiga keluarga pendekatan:

1. **Linear** — Logistic Regression (baseline v1, sebagai pembanding).
2. **Tree ensemble / boosting** — Random Forest, HistGradientBoosting, XGBoost, LightGBM, CatBoost.
3. **Margin-based** — SVM RBF (probabilitas via Platt scaling).

Model boosting diharapkan menangkap hubungan **non-linear** (mis. lonjakan turnover pada jam lembur tinggi) tanpa perlu fitur interaksi manual.

In [4]:
def build_models() -> dict:
    return {
        "Dummy (prior)": DummyClassifier(strategy="prior"),
        "Logistic Regression (v1)": LogisticRegression(max_iter=2000, random_state=SEED),
        "Random Forest": RandomForestClassifier(
            n_estimators=400, min_samples_leaf=3, random_state=SEED, n_jobs=1),
        "Hist Gradient Boosting": HistGradientBoostingClassifier(random_state=SEED),
        "XGBoost": XGBClassifier(
            n_estimators=400, learning_rate=0.05, max_depth=3, subsample=0.8,
            colsample_bytree=0.8, reg_lambda=1.0, eval_metric="logloss",
            random_state=SEED, n_jobs=1),
        "LightGBM": LGBMClassifier(
            n_estimators=400, learning_rate=0.05, num_leaves=15,
            min_child_samples=20, random_state=SEED, n_jobs=1, verbose=-1),
        "CatBoost": CatBoostClassifier(
            iterations=400, learning_rate=0.05, depth=4, random_seed=SEED,
            verbose=0, thread_count=1),
        "SVM (RBF)": SVC(probability=True, random_state=SEED),
    }


def to_pipe(estimator) -> Pipeline:
    return Pipeline([("prep", make_preprocessor()), ("clf", clone(estimator))])


list(build_models().keys())

['Dummy (prior)',
 'Logistic Regression (v1)',
 'Random Forest',
 'Hist Gradient Boosting',
 'XGBoost',
 'LightGBM',
 'CatBoost',
 'SVM (RBF)']

## 4. Perbandingan Model dengan Cross-Validation

Menggunakan `RepeatedStratifiedKFold(5, 3)` (15 fit) pada **data train saja**.
Metrik utama **ROC-AUC** (tidak bergantung threshold), sekunder **PR-AUC** (cocok untuk kelas tidak seimbang).

In [5]:
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "recall": "recall",
    "precision": "precision",
    "f1": "f1",
}
cv_repeat = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)

cv_mean, cv_std = {}, {}
for name, est in build_models().items():
    res = cross_validate(to_pipe(est), X_train, y_train, cv=cv_repeat, scoring=scoring, n_jobs=-1)
    cv_mean[name] = {m: res[f"test_{m}"].mean() for m in scoring}
    cv_std[name] = {m: res[f"test_{m}"].std() for m in scoring}

cv_mean = pd.DataFrame(cv_mean).T.sort_values("roc_auc", ascending=False)
cv_std = pd.DataFrame(cv_std).T.loc[cv_mean.index]

cv_table = cv_mean.copy()
for m in scoring:
    cv_table[m] = [f"{cv_mean.loc[i, m]:.3f} ± {cv_std.loc[i, m]:.3f}" for i in cv_mean.index]
cv_table

,roc_auc,pr_auc,recall,precision,f1
Logistic Regression (v1),0.690 ± 0.048,0.629 ± 0.055,0.393 ± 0.077,0.613 ± 0.066,0.476 ± 0.072
Random Forest,0.684 ± 0.037,0.617 ± 0.053,0.381 ± 0.065,0.666 ± 0.071,0.482 ± 0.060
SVM (RBF),0.670 ± 0.048,0.613 ± 0.046,0.276 ± 0.047,0.716 ± 0.071,0.397 ± 0.058
CatBoost,0.670 ± 0.051,0.610 ± 0.051,0.396 ± 0.052,0.607 ± 0.066,0.477 ± 0.046
LightGBM,0.653 ± 0.038,0.581 ± 0.050,0.433 ± 0.059,0.543 ± 0.054,0.480 ± 0.050
XGBoost,0.651 ± 0.043,0.587 ± 0.056,0.399 ± 0.051,0.563 ± 0.061,0.465 ± 0.046
Hist Gradient Boosting,0.651 ± 0.043,0.589 ± 0.052,0.430 ± 0.067,0.572 ± 0.054,0.489 ± 0.058
Dummy (prior),0.500 ± 0.000,0.356 ± 0.000,0.000 ± 0.000,0.000 ± 0.000,0.000 ± 0.000


In [6]:
kandidat = cv_mean.drop(index="Dummy (prior)")
kandidat[["roc_auc", "pr_auc"]].plot.barh(figsize=(8, 4), color=["#4C78A8", "#F58518"])
plt.axvline(0.5, ls="--", color="grey", lw=1, label="acak (ROC-AUC)")
plt.xlabel("skor CV (rata-rata)")
plt.title("Perbandingan metode — ROC-AUC & PR-AUC")
plt.legend()
plt.tight_layout()
plt.show()

selisih = (kandidat["roc_auc"].max() - kandidat["roc_auc"].min())
print(f"Rentang ROC-AUC antar metode: {selisih:.3f}")
print(f"Metode terbaik di CV        : {kandidat['roc_auc'].idxmax()} "
      f"({kandidat['roc_auc'].max():.3f})")

Rentang ROC-AUC antar metode: 0.039
Metode terbaik di CV        : Logistic Regression (v1) (0.690)


## 5. Hyperparameter Tuning

`RandomizedSearchCV` (ROC-AUC) hanya untuk **model teratas** dari tahap sebelumnya.
Ruang pencarian disesuaikan per algoritma. Test set belum disentuh.

In [7]:
PARAM_SPACES = {
    "Logistic Regression (v1)": {
        "clf__C": loguniform(1e-3, 1e2),
    },
    "Random Forest": {
        "clf__n_estimators": randint(300, 800),
        "clf__max_depth": [None, 4, 6, 8, 12],
        "clf__min_samples_leaf": randint(1, 15),
        "clf__max_features": ["sqrt", "log2", 0.5],
    },
    "Hist Gradient Boosting": {
        "clf__learning_rate": loguniform(0.01, 0.3),
        "clf__max_iter": randint(100, 400),
        "clf__max_leaf_nodes": randint(4, 32),
        "clf__min_samples_leaf": randint(10, 60),
        "clf__l2_regularization": loguniform(1e-3, 10),
        "clf__max_depth": [None, 3, 4, 6],
    },
    "XGBoost": {
        "clf__n_estimators": randint(200, 700),
        "clf__learning_rate": loguniform(0.01, 0.3),
        "clf__max_depth": randint(2, 7),
        "clf__subsample": loguniform(0.6, 1.0),
        "clf__colsample_bytree": loguniform(0.6, 1.0),
        "clf__reg_lambda": loguniform(1e-3, 10),
    },
    "LightGBM": {
        "clf__n_estimators": randint(200, 700),
        "clf__learning_rate": loguniform(0.01, 0.3),
        "clf__num_leaves": randint(4, 40),
        "clf__min_child_samples": randint(5, 40),
        "clf__reg_lambda": loguniform(1e-3, 10),
    },
    "CatBoost": {
        "clf__iterations": randint(200, 700),
        "clf__learning_rate": loguniform(0.01, 0.3),
        "clf__depth": randint(3, 7),
        "clf__l2_leaf_reg": loguniform(1, 10),
    },
    "SVM (RBF)": {
        "clf__C": loguniform(0.1, 100),
        "clf__gamma": loguniform(1e-3, 1),
    },
}

top_names = list(kandidat["roc_auc"].sort_values(ascending=False).index[:TUNE_TOP_N])
print("Model yang di-tuning:", top_names)

cv_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
tuned = {}
tuning_log = []

for name in top_names:
    search = RandomizedSearchCV(
        to_pipe(build_models()[name]),
        param_distributions=PARAM_SPACES[name],
        n_iter=25,
        scoring="roc_auc",
        cv=cv_tune,
        n_jobs=-1,
        random_state=SEED,
        refit=True,
    )
    search.fit(X_train, y_train)
    tuned[name] = search.best_estimator_
    tuning_log.append({
        "model": name,
        "roc_auc_default": round(float(kandidat.loc[name, "roc_auc"]), 4),
        "roc_auc_tuned": round(float(search.best_score_), 4),
        "params": {k.replace("clf__", ""): (round(v, 5) if isinstance(v, float) else v)
                   for k, v in search.best_params_.items()},
    })
    print(f"{name:<26} default {kandidat.loc[name, 'roc_auc']:.4f} -> tuned {search.best_score_:.4f}")

pd.DataFrame(tuning_log).set_index("model")

Model yang di-tuning: ['Logistic Regression (v1)', 'Random Forest']


Logistic Regression (v1)   default 0.6900 -> tuned 0.6971


Random Forest              default 0.6838 -> tuned 0.7050


,roc_auc_default,roc_auc_tuned,params
model,,,
Logistic Regression (v1),0.6900,0.6971,{'C': 0.00603}
Random Forest,0.6838,0.7050,"{'max_depth': 4, 'max_features': 'log2', 'min_..."


## 6. Ensemble (Soft Voting)

Menggabungkan model-model yang sudah di-tuning. **Soft voting** merata-ratakan probabilitas,
sehingga prediksi lebih stabil dan sering menurunkan varians tanpa mengorbankan AUC.

In [8]:
voting_estimators = [(name, clone(pipe.named_steps["clf"])) for name, pipe in tuned.items()]
voter = VotingClassifier(estimators=voting_estimators, voting="soft")
voting_pipe = Pipeline([("prep", make_preprocessor()), ("clf", voter)])

res_vote = cross_validate(voting_pipe, X_train, y_train, cv=cv_tune, scoring=scoring, n_jobs=-1)
vote_scores = {m: res_vote[f"test_{m}"].mean() for m in scoring}
print("Soft Voting (CV):")
for m, v in vote_scores.items():
    print(f"  {m:<10}: {v:.4f} ± {res_vote[f'test_{m}'].std():.4f}")

Soft Voting (CV):
  roc_auc   : 0.7085 ± 0.0456
  pr_auc    : 0.6495 ± 0.0727
  recall    : 0.2491 ± 0.0537
  precision : 0.8116 ± 0.1432
  f1        : 0.3808 ± 0.0783


## 7. Pemilihan Model Juara

Juara dipilih berdasarkan **ROC-AUC cross-validation** (bukan test set).
Kandidat = model terbaik hasil tuning **vs** soft-voting ensemble.

In [9]:
challengers = {name: float(tuning_log[i]["roc_auc_tuned"]) for i, name in enumerate(top_names)}
challengers["Soft Voting (ensemble)"] = vote_scores["roc_auc"]

leaderboard = pd.Series(challengers, name="roc_auc_cv").sort_values(ascending=False).to_frame()
leaderboard["pr_auc_cv"] = [
    cv_mean.loc[name, "pr_auc"] if name in cv_mean.index else vote_scores["pr_auc"]
    for name in leaderboard.index
]
champion_name = leaderboard.index[0]

if champion_name == "Soft Voting (ensemble)":
    champion_pipe = voting_pipe
else:
    champion_pipe = tuned[champion_name]

# Fit juara pada data train (ensemble belum di-fit; model tuned sudah, refit aman).
champion_pipe.fit(X_train, y_train)

print("Peringkat akhir (CV):")
print(leaderboard.round(4).to_string())
print(f"\n>>> Juara: {champion_name}")

Peringkat akhir (CV):
                          roc_auc_cv  pr_auc_cv
Soft Voting (ensemble)        0.7085     0.6495
Random Forest                 0.7050     0.6166
Logistic Regression (v1)      0.6971     0.6290

>>> Juara: Soft Voting (ensemble)


## 8. Pemilihan Threshold (Out-of-Fold)

Threshold dipilih dari prediksi **out-of-fold pada data train** dengan kriteria memaksimalkan F1 —
sama seperti v1, agar metrik threshold-dependent bisa dibandingkan.

In [10]:
oof_proba = cross_val_predict(
    clone(champion_pipe), X_train, y_train,
    cv=cv_tune, method="predict_proba", n_jobs=-1,
)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
tabel_thr = pd.DataFrame({
    "threshold": thresholds,
    "precision": [precision_score(y_train, oof_proba >= t, zero_division=0) for t in thresholds],
    "recall": [recall_score(y_train, oof_proba >= t) for t in thresholds],
    "f1": [f1_score(y_train, oof_proba >= t, zero_division=0) for t in thresholds],
})

BEST_THRESHOLD = round(float(tabel_thr.loc[tabel_thr["f1"].idxmax(), "threshold"]), 2)
print(f"Threshold terpilih (maks F1 pada OOF): {BEST_THRESHOLD:.2f}")

fig, ax = plt.subplots(figsize=(8, 4))
for m, c in zip(["precision", "recall", "f1"], ["#4C78A8", "#E45756", "#54A24B"]):
    ax.plot(tabel_thr["threshold"], tabel_thr[m], label=m, color=c)
ax.axvline(BEST_THRESHOLD, ls="--", color="black", lw=1, label=f"terpilih = {BEST_THRESHOLD:.2f}")
ax.set_xlabel("threshold")
ax.set_ylabel("skor (OOF, data train)")
ax.set_title("Trade-off precision–recall terhadap threshold")
ax.legend()
plt.tight_layout()
plt.show()

Threshold terpilih (maks F1 pada OOF): 0.33


## 9. Evaluasi Final pada Test Set

Test set dibuka **sekali** di sini.

In [11]:
proba_test = champion_pipe.predict_proba(X_test)[:, 1]
proba_train = champion_pipe.predict_proba(X_train)[:, 1]
y_te = y_test.to_numpy()


def ringkas_metrik(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {
        "accuracy": accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred),
        "f1": f1_score(y_true, pred, zero_division=0),
    }


metrik_test = {
    "roc_auc": roc_auc_score(y_te, proba_test),
    "pr_auc": average_precision_score(y_te, proba_test),
    "brier": brier_score_loss(y_te, proba_test),
    **ringkas_metrik(y_te, proba_test, BEST_THRESHOLD),
}

hasil = pd.DataFrame({
    "threshold 0.50": ringkas_metrik(y_te, proba_test, 0.5),
    f"threshold {BEST_THRESHOLD:.2f} (terpilih)": ringkas_metrik(y_te, proba_test, BEST_THRESHOLD),
}).T.round(3)

print(f"=== Metrik di TEST SET — {champion_name} ===")
print(f"ROC-AUC : {metrik_test['roc_auc']:.3f}")
print(f"PR-AUC  : {metrik_test['pr_auc']:.3f}  (baseline acak = {y_te.mean():.3f})")
print(f"Brier   : {metrik_test['brier']:.3f}  (makin kecil makin baik)")
print()
print(hasil.to_string())

rng = np.random.default_rng(SEED)
boot = []
for _ in range(1000):
    idx = rng.integers(0, len(y_te), len(y_te))
    if len(np.unique(y_te[idx])) < 2:
        continue
    boot.append(roc_auc_score(y_te[idx], proba_test[idx]))
ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
print(f"\nROC-AUC test 95% CI (bootstrap): [{ci_low:.3f}, {ci_high:.3f}]")

print("\n=== Cek overfitting (ROC-AUC) ===")
print(f"train : {roc_auc_score(y_train, proba_train):.3f}")
print(f"test  : {metrik_test['roc_auc']:.3f}")

=== Metrik di TEST SET — Soft Voting (ensemble) ===
ROC-AUC : 0.699
PR-AUC  : 0.622  (baseline acak = 0.355)
Brier   : 0.203  (makin kecil makin baik)

                           accuracy  precision  recall     f1
threshold 0.50                 0.71      0.741   0.282  0.408
threshold 0.33 (terpilih)      0.62      0.475   0.676  0.558



ROC-AUC test 95% CI (bootstrap): [0.622, 0.774]

=== Cek overfitting (ROC-AUC) ===
train : 0.766
test  : 0.699


In [12]:
print(classification_report(
    y_te, (proba_test >= BEST_THRESHOLD).astype(int),
    target_names=["bertahan (0)", "keluar (1)"], digits=3,
))

from sklearn.metrics import (
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
RocCurveDisplay.from_predictions(y_te, proba_test, ax=axes[0], name=champion_name)
axes[0].plot([0, 1], [0, 1], "--", color="grey", label="acak")
axes[0].set_title("Kurva ROC (test)")
axes[0].legend(loc="lower right")

PrecisionRecallDisplay.from_predictions(y_te, proba_test, ax=axes[1], name=champion_name)
axes[1].axhline(y_te.mean(), ls="--", color="grey", label="baseline (acak)")
axes[1].set_title("Kurva Precision–Recall (test)")
axes[1].legend(loc="lower left")

ConfusionMatrixDisplay.from_predictions(
    y_te, (proba_test >= BEST_THRESHOLD).astype(int),
    display_labels=["bertahan", "keluar"], cmap="Blues", ax=axes[2], colorbar=False)
axes[2].set_title(f"Confusion matrix (threshold {BEST_THRESHOLD:.2f})")
axes[2].grid(False)
plt.tight_layout()
plt.show()

              precision    recall  f1-score   support

bertahan (0)      0.768     0.589     0.667       129
  keluar (1)      0.475     0.676     0.558        71

    accuracy                          0.620       200
   macro avg      0.621     0.633     0.612       200
weighted avg      0.664     0.620     0.628       200



In [13]:
frac_pos, mean_pred = calibration_curve(y_te, proba_test, n_bins=8, strategy="quantile")

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "--", color="grey", label="terkalibrasi sempurna")
ax.plot(mean_pred, frac_pos, "o-", color="#E45756", label=champion_name)
ax.set_xlabel("probabilitas prediksi (rata-rata per bin)")
ax.set_ylabel("proporsi aktual keluar")
ax.set_title(f"Kurva kalibrasi (test) — Brier {metrik_test['brier']:.3f}")
ax.legend()
plt.tight_layout()
plt.show()

## 10. Interpretasi Model

Dua sudut pandang:
- **Permutation importance** — seberapa besar ROC-AUC turun saat fitur diacak (model-agnostic).
- **SHAP** — kontribusi tiap fitur per baris (arah pengaruh, bukan hanya besar).

In [14]:
perm = permutation_importance(
    champion_pipe, X_test, y_test, scoring="roc_auc",
    n_repeats=30, random_state=SEED, n_jobs=-1,
)
imp = (pd.DataFrame({
    "fitur": X_test.columns,
    "penurunan_auc": perm.importances_mean,
    "std": perm.importances_std,
}).sort_values("penurunan_auc", ascending=True))

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(imp["fitur"], imp["penurunan_auc"], xerr=imp["std"], color="#4C78A8")
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("penurunan ROC-AUC saat fitur diacak")
ax.set_title("Permutation importance (test set)")
plt.tight_layout()
plt.show()

imp.sort_values("penurunan_auc", ascending=False).head(6).round(4).reset_index(drop=True)

,fitur,penurunan_auc,std
0,jam_lembur_bulanan,0.1851,0.0342
1,status_kerja,0.0375,0.0125
2,pendidikan,0.0104,0.0055
3,usia_saat_masuk,0.0056,0.0036
4,ada_promosi,0.0052,0.0041
5,log_gaji,0.0030,0.0144


In [15]:
import shap

prep = champion_pipe.named_steps["prep"]
clf = champion_pipe.named_steps["clf"]
X_test_t = prep.transform(X_test)
feature_names = prep.get_feature_names_out()

try:
    explainer = shap.TreeExplainer(clf)
    shap_values = explainer.shap_values(X_test_t)
    if isinstance(shap_values, list):  # beberapa model mengembalikan list per kelas
        shap_values = shap_values[1]
    shap.summary_plot(shap_values, X_test_t, feature_names=feature_names, show=True)
except Exception as exc:  # fallback untuk model non-tree (linear/SVM/voting)
    print(f"TreeExplainer tidak tersedia untuk {champion_name} ({exc}).")
    print("Menampilkan ringkasan berbasis permutation importance saja.")

TreeExplainer tidak tersedia untuk Soft Voting (ensemble) (Model type not yet supported by TreeExplainer: <class 'sklearn.ensemble._voting.VotingClassifier'>).
Menampilkan ringkasan berbasis permutation importance saja.


## 11. Simpan Model, Metadata, dan Contoh Inferensi

Artifact disimpan ke `models/v2/`. Skema metadata **tetap kompatibel** dengan
`hr_intelligence_api` (kunci: `model`, `dibuat_pada`, `threshold`, `fitur_numerik`,
`fitur_kategorik`, `kolom_mentah_dibutuhkan`, `tanggal_acuan`, `metrik_holdout_test`, `catatan`),
plus tambahan `versi_model` dan `perbandingan_model`.

In [16]:
if REFIT_ON_FULL_DATA:
    model_final = clone(champion_pipe).fit(X, y)
    print(f"Model final dilatih ulang pada seluruh data ({len(X)} baris).")
else:
    model_final = champion_pipe
    print("Model final = model yang dilatih di data train saja.")

MODEL_PATH = ARTIFACT_DIR / "turnover_model.joblib"
META_PATH = ARTIFACT_DIR / "model_metadata.json"

if champion_name == "Soft Voting (ensemble)":
    hyperparams = {n: {k.replace("clf__", ""): v for k, v in p.named_steps["clf"].get_params().items()
                       if k.startswith(("n_estimators", "learning_rate", "max_depth", "C", "num_leaves",
                                        "depth", "iterations", "subsample", "colsample_bytree",
                                        "reg_lambda", "l2_leaf_reg", "min_child_samples", "gamma",
                                        "max_features", "min_samples_leaf", "max_leaf_nodes",
                                        "max_iter", "l2_regularization"))}
                   for n, p in tuned.items()}
else:
    hyperparams = {k.replace("clf__", ""): v for k, v in champion_pipe.named_steps["clf"].get_params().items()
                   if k.startswith(("C", "n_estimators", "learning_rate", "max_depth", "num_leaves",
                                    "depth", "iterations", "subsample", "colsample_bytree",
                                    "reg_lambda", "l2_leaf_reg", "min_child_samples", "gamma",
                                    "max_features", "min_samples_leaf", "max_leaf_nodes",
                                    "max_iter", "l2_regularization"))}

metadata = {
    "versi_model": MODEL_VERSION,
    "dibuat_pada": datetime.now().isoformat(timespec="seconds"),
    "tujuan": "Prediksi turnover karyawan (1 = keluar)",
    "sumber_data": DATA_PATH.name,
    "jumlah_baris": int(len(df_raw)),
    "rasio_positif": round(float(y.mean()), 4),
    "model": champion_name,
    "metode": "alternatif_v2",
    "hyperparameter": hyperparams,
    "threshold": BEST_THRESHOLD,
    "refit_seluruh_data": REFIT_ON_FULL_DATA,
    "fitur_numerik": NUM_COLS,
    "fitur_kategorik": CAT_COLS,
    "kolom_mentah_dibutuhkan": [
        "usia", "masa_kerja_bulan", "jam_lembur_bulanan", "gaji_bulanan",
        "tanggal_masuk", "tanggal_promosi", "departemen", "status_kerja", "pendidikan",
    ],
    "tanggal_acuan": str(REF_DATE.date()),
    "metrik_holdout_test": {
        **{k: round(float(v), 4) for k, v in metrik_test.items()},
        "roc_auc_95ci": [round(float(ci_low), 4), round(float(ci_high), 4)],
    },
    "roc_auc_cv_tuned": round(float(leaderboard.loc[champion_name, "roc_auc_cv"]), 4),
    "perbandingan_model": {
        name: {"roc_auc_cv": round(float(cv_mean.loc[name, "roc_auc"]), 4),
               "pr_auc_cv": round(float(cv_mean.loc[name, "pr_auc"]), 4)}
        for name in cv_mean.index
    },
    "versi": {
        "python_sklearn": sklearn.__version__,
        "pandas": pd.__version__,
        "numpy": np.__version__,
    },
    "catatan": (
        "Model v2 (metode alternatif). Gunakan buat_fitur() yang sama "
        "untuk mengubah data mentah menjadi fitur sebelum predict."
    ),
}

joblib.dump(model_final, MODEL_PATH)
META_PATH.write_text(json.dumps(metadata, indent=2, ensure_ascii=False, default=str), encoding="utf-8")

print(f"Model   : {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1024:.0f} KB)")
print(f"Metadata: {META_PATH}")
print(f"Juara   : {champion_name} | threshold {BEST_THRESHOLD:.2f}")

Model final dilatih ulang pada seluruh data (1000 baris).
Model   : /workspace/project/hr-intelligence-system/models/v2/turnover_model.joblib  (1395 KB)
Metadata: /workspace/project/hr-intelligence-system/models/v2/model_metadata.json
Juara   : Soft Voting (ensemble) | threshold 0.33


In [17]:
model_loaded = joblib.load(MODEL_PATH)
meta_loaded = json.loads(META_PATH.read_text(encoding="utf-8"))


def prediksi_turnover(df_baru: pd.DataFrame, model=model_loaded, meta=meta_loaded) -> pd.DataFrame:
    """Terima data mentah (kolom seperti CSV), kembalikan probabilitas & level risiko."""
    perlu = meta["kolom_mentah_dibutuhkan"]
    kurang = [c for c in perlu if c not in df_baru.columns]
    if kurang:
        raise ValueError(f"Kolom wajib belum ada: {kurang}")

    proba = model.predict_proba(buat_fitur(df_baru, pd.Timestamp(meta["tanggal_acuan"])))[:, 1]
    hasil = pd.DataFrame({
        ID_COL: df_baru[ID_COL].values,
        "probabilitas_keluar": proba.round(4),
        "prediksi_keluar": (proba >= meta["threshold"]).astype(int),
    })
    hasil["level_risiko"] = pd.cut(
        proba, [-0.01, 0.25, 0.50, 1.01], labels=["Rendah", "Sedang", "Tinggi"])
    return hasil


contoh = df_raw.sample(8, random_state=1)
out = prediksi_turnover(contoh)
out["aktual"] = contoh[TARGET].values
print("Model termuat :", meta_loaded["model"], "| versi:", meta_loaded["versi_model"],
      "| threshold =", meta_loaded["threshold"])
out

Model termuat : Soft Voting (ensemble) | versi: v2 | threshold = 0.33


,employee_id,probabilitas_keluar,prediksi_keluar,level_risiko,aktual
0,EMP0508,0.3046,0,Sedang,0
1,EMP0819,0.3566,1,Sedang,0
2,EMP0453,0.5355,1,Tinggi,1
3,EMP0369,0.5816,1,Tinggi,1
4,EMP0243,0.3442,1,Sedang,1
5,EMP0930,0.3452,1,Sedang,0
6,EMP0263,0.3027,0,Sedang,0
7,EMP0811,0.2499,0,Rendah,0


In [18]:
# Verifikasi model di disk menghasilkan prediksi identik dengan model di memori
p_mem = model_final.predict_proba(X_test)[:, 1]
p_disk = model_loaded.predict_proba(X_test)[:, 1]
print("Prediksi model tersimpan identik dengan model di memori:", np.allclose(p_mem, p_disk))

Prediksi model tersimpan identik dengan model di memori: True


## Ringkasan & Perbandingan dengan v1

| Aspek | v1 | v2 (notebook ini) |
|---|---|---|
| Algoritma | Logistic Regression | lihat `metadata["model"]` |
| ROC-AUC CV | 0.6971 | lihat `metadata["roc_auc_cv_tuned"]` |
| ROC-AUC test | 0.6929 | lihat `metadata["metrik_holdout_test"]["roc_auc"]` |
| PR-AUC test | 0.6235 | lihat `metadata["metrik_holdout_test"]["pr_auc"]` |
| Artifact | `models/v1/` | `models/v2/` |

Kesimpulan praktis dari perbandingan ini:
- Pada dataset kecil (1.000 baris) dengan sinyal yang sebagian besar **monoton** (jam lembur, masa kerja),
  model linear yang ter-regularisasi sangat kompetitif. Boosting menang terutama bila ada interaksi non-linear.
- **Kalau selisih ROC-AUC antar metode lebih kecil dari lebar confidence interval**, jangan pilih yang rumit —
  pilih yang lebih sederhana & lebih mudah dijelaskan ke stakeholder HR.

### Langkah lanjut yang disarankan
1. Bandingkan juga di **data yang lebih besar / lebih baru** sebelum mengganti model produksi.
2. Jalankan **uji bias per kelompok** (usia, departemen, status kerja) sebelum dipakai untuk keputusan retensi.
3. Bila ada tanggal keluar, naikkan ke **survival analysis** (Cox / gradient boosting survival).
4. Tambahkan **monitoring drift** dan retraining berkala (lihat folder `mlops/`).

> **Catatan etika:** hasil prediksi dipakai sebagai skor prioritas untuk percakapan retensi,
> **bukan** dasar keputusan yang merugikan karyawan (PHK, penolakan promosi).